# 1. State Reducer

## 1.1 什么是 Reducer

Graph 运行时把 State 按 **key（channel）** 逐个合并。Node 返回的 dict 就是「State 更新」，同一个 key 被多个 Node 写入时，怎么合并由该 key 的 **reducer** 决定：

- **不加 reducer**：后写直接覆盖先写（last-writer-wins）
- **加 reducer**：`新值 = reducer(当前值, 更新值)`，Node 可以只返回增量

reducer 就是一个普通函数：`(当前值, 更新值) -> 新值`，通过 `Annotated[T, reducer]` 挂到字段上。

> 补充：多个 Node **并行**写同一个无 reducer 的 key 会直接报错（一个 super-step 里一个 key 只能收一个值）；带 reducer 的 key 允许多个写入者，各自返回增量即可。

下面用同一个 Graph、两种 schema 对比：

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


# 两个 Node 连续写同一个 key
def node_inc(state) -> dict:
    return {"count": state["count"] + 1}


def node_add10(state) -> dict:
    return {"count": 10}


# 1) 无 reducer：后写覆盖
class OverwriteState(TypedDict):
    count: int


builder = StateGraph(state_schema=OverwriteState)
builder.add_node("inc", node_inc)
builder.add_node("add10", node_add10)
builder.add_edge(START, "inc")
builder.add_edge("inc", "add10")
builder.add_edge("add10", END)
result = builder.compile().invoke({"count": 0})
print("无 reducer（后写覆盖）:", result["count"])


# 2) 带 reducer：累加合并
class AddState(TypedDict):
    count: Annotated[int, add]


builder = StateGraph(state_schema=AddState)
builder.add_node("inc", node_inc)
builder.add_node("add10", node_add10)
builder.add_edge(START, "inc")
builder.add_edge("inc", "add10")
builder.add_edge("add10", END)
result = builder.compile().invoke({"count": 0})
print("带 reducer（累加合并）:", result["count"])
# 无 reducer（后写覆盖）: 10
# 带 reducer（累加合并）: 11

## 1.2 自定义 Reducer

自定义 reducer 只需写一个 `(current, update) -> new` 的纯函数，再用 `Annotated[T, fn]` 挂到字段上：

- 第一个参数是旧值，第二个参数是更新值
- **必须是纯函数**：不要做 IO、不要依赖外部可变状态（重放和并行场景会多次调用）
- 并行 Node 同时写同一个带 reducer 的 key 时，更新按任务顺序链式合并，结果要与顺序无关

下面演示两个自定义 reducer：dict 合并、取最大值：

In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


# 自定义 reducer：dict 合并 / 取最大值
def merge_dict(a: dict, b: dict) -> dict:
    return {**a, **b}


def take_max(a: float, b: float) -> float:
    return max(a, b)


class ScoreState(TypedDict):
    cfg: Annotated[dict, merge_dict]
    score: Annotated[float, take_max]


# 两个并行 Node 各自写一部分
def node_a(state: ScoreState) -> dict:
    return {"cfg": {"model": "gpt-4"}, "score": 0.7}


def node_b(state: ScoreState) -> dict:
    return {"cfg": {"temperature": 0.2}, "score": 0.9}


builder = StateGraph(state_schema=ScoreState)
builder.add_node("a", node_a)
builder.add_node("b", node_b)
builder.add_edge(START, "a")
builder.add_edge(START, "b")
builder.add_edge("a", END)
builder.add_edge("b", END)

print(builder.compile().invoke({"cfg": {}, "score": 0}))
# {'cfg': {'model': 'gpt-4', 'temperature': 0.2}, 'score': 0.9}

## 1.3 内置 Reducer

LangGraph 内置了几个常用 reducer：

| Reducer | 适用字段 | 行为 |
|---|---|---|
| `operator.add` | `list` / `int` | list **拼接**；int **相加**（计数器） |
| `add_messages` | message 列表 | 按 `message.id` 去重：同 id **替换**，新 id **追加**；`RemoveMessage(id=...)` 删除 |
| `Overwrite(value)` | 任意带 reducer 的字段 | 绕过 reducer **直接覆盖**；同一 super-step 内多个 `Overwrite` 会报错 |

`add_messages`（`from langgraph.graph.message import add_messages`）是聊天场景的核心，LangGraph 还提供了现成的 `MessagesState`；`Overwrite` 从 `langgraph.types` 导入。

下面逐个看它们的运作逻辑。

### 1.3.1 `operator.add`：list 拼接 / int 相加

`operator.add` 是最简单的 reducer：**新值 = 旧值 + 更新值**。list 是**拼接**，int 是**相加**（适合做计数器）。

先看 list 拼接：两个 Node 先后往同一个 key 追加日志。

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class LogState(TypedDict):
    logs: Annotated[list[str], add]


def log_a(state: LogState) -> dict:
    return {"logs": ["node_a 运行"]}


def log_b(state: LogState) -> dict:
    return {"logs": ["node_b 运行"]}


builder = StateGraph(state_schema=LogState)
builder.add_node("a", log_a)
builder.add_node("b", log_b)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)

print(builder.compile().invoke({"logs": []}))
# {'logs': ['node_a 运行', 'node_b 运行']}

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


# 同样的 add 挂到 int 字段上，就是计数器
class CountState(TypedDict):
    count: Annotated[int, add]


def inc_1(state: CountState) -> dict:
    return {"count": 1}


def inc_10(state: CountState) -> dict:
    return {"count": 10}


builder = StateGraph(state_schema=CountState)
builder.add_node("inc_1", inc_1)
builder.add_node("inc_10", inc_10)
builder.add_edge(START, "inc_1")
builder.add_edge("inc_1", "inc_10")
builder.add_edge("inc_10", END)

print(builder.compile().invoke({"count": 0}))
# {'count': 11}

### 1.3.2 `add_messages`：按 id 去重合并

聊天场景专用 reducer，按 `message.id` 判断：

- **同 id** → 新消息**替换**旧消息
- **新 id** → **追加**到尾部
- `RemoveMessage(id=...)` → 按 id **删除**

In [ ]:
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, RemoveMessage
from langgraph.graph.message import add_messages

base: list[AnyMessage] = [HumanMessage(content="你好", id="1")]
update: list[AnyMessage] = [
    HumanMessage(content="你好呀", id="1"),  # 同 id → 替换
    AIMessage(content="有什么可以帮你？", id="2"),  # 新 id → 追加
]
# add_messages 签名类型较宽，这里显式收窄为 list[AnyMessage] 
merged: list[AnyMessage] = add_messages(base, update)  # type: ignore
print("合并结果:", [m.content for m in merged])
# 合并结果: ['你好呀', '有什么可以帮你？']

# 按 id 删除
after_delete: list[AnyMessage] = add_messages(merged, [RemoveMessage(id="1")])  # type: ignore
print("删除后:", [m.content for m in after_delete])
# 删除后: ['有什么可以帮你？']

### 1.3.3 `Overwrite`：绕过 reducer 直接覆盖

想**整体重置**某个字段时（比如一键清空对话），把更新值用 `Overwrite(value)` 包一层，LangGraph 会跳过 reducer，直接把字段写成 `value`。

注意：同一 super-step 里多个 Node 对同一个 key 写 `Overwrite` 会直接报错——整体覆盖无法合并，只能有一个写入者。

下面用现成的 `MessagesState` 演示清空对话：

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Overwrite
from IPython.display import display


# MessagesState = {"messages": Annotated[list[AnyMessage], add_messages]}
def node_ai(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="我是助手")]}


def node_1(state: MessagesState) -> dict:
    return {"messages": [HumanMessage(content="今天天气怎么样？")]}


def node_2(state: MessagesState) -> dict:
    return {"messages": [HumanMessage(content="今天准备做什么？")]}


def clear(state: MessagesState) -> dict:
    # 直接返回会走 add_messages 追加；Overwrite 包一层 → 整体替换
    return {"messages": Overwrite([SystemMessage(content="对话已清空")])}


builder = StateGraph(state_schema=MessagesState)
builder.add_node("node_ai", node_ai)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("clear", clear)
builder.add_edge(START, "node_ai")
builder.add_edge("node_ai", "node_1")
builder.add_edge("node_ai", "node_2")
builder.add_edge("node_ai", "clear")
builder.add_edge("clear", END)
builder.add_edge("node_1", END)
builder.add_edge("node_2", END)

graph = builder.compile()
display(graph)


result = graph.invoke({"messages": [HumanMessage(content="你好")]})
print("清空后:", [m.content for m in result["messages"]])
# 清空后: ['对话已清空']

## 1.4 Pydantic State

第 1 章说过 State 可以是「TypedDict **或** Pydantic 类」。前几节都用 TypedDict，这里补上 Pydantic（`BaseModel`）版本，并说清两者差异：

| 维度 | TypedDict | Pydantic BaseModel |
|---|---|---|
| Node 收到的 State | dict（下标访问 `state["user"]`） | **模型实例**（属性访问 `state.user`；下标访问直接 `TypeError`） |
| 入参校验 | 无（多写/少写/写错类型都不报错） | 有：缺必填字段、类型不符 → 入口处抛 `ValidationError` |
| 默认值 | 无（key 必须显式传入，否则 `KeyError`） | 有（`Field(default_factory=list)`），invoke 可省略 |
| 适用 | 轻量内部流程 | 对外暴露的 Graph / 需要严格校验的场景 |

注意：

- **可变默认值必须用 `default_factory`**（`Field(default_factory=list)`），直接 `= []` 会被所有实例共享
- reducer 用法不变：还是 `Annotated[list[str], add]`
- Node 的返回值依然是 **dict**（部分更新），invoke 返回的也是 dict——只有「Node 入参」是模型实例

选型建议：学习/内部流程用 TypedDict 更轻；Graph 对外当服务提供（入参来自不可信调用方）时用 Pydantic，把脏数据挡在入口。

In [ ]:
from operator import add
from typing import Annotated

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field, ValidationError


class ChatState(BaseModel):
    logs: Annotated[list[str], add] = Field(default_factory=list)  # 可变默认值必须 default_factory
    user: str = ""


def greet(state: ChatState) -> dict:   # 入参是模型实例：属性访问
    print("属性访问 state.user:", state.user)
    print("模型实例:", type(state).__name__)
    return {"logs": [f"hello {state.user}"]}   # 返回值仍是 dict 部分更新


builder = StateGraph(state_schema=ChatState)
builder.add_node("greet", greet)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

print(builder.compile().invoke({"user": "Alice"}))   # logs 有默认值，可不传
# 属性访问 state.user: Alice
# 模型实例: ChatState
# {'logs': ['hello Alice'], 'user': 'Alice'}   ← invoke 返回的是 dict


# 入参校验：类型不符 → 入口直接 ValidationError（TypedDict 不会报错）
class Strict(BaseModel):
    n: int


builder2 = StateGraph(state_schema=Strict)
builder2.add_node("n1", lambda s: {})
builder2.add_edge(START, "n1")
builder2.add_edge("n1", END)

try:
    builder2.compile().invoke({"n": "not-an-int"})
except ValidationError as e:
    print(f"{type(e).__name__}: {str(e)[:70]}")
# ValidationError: 1 validation error for Strict